## Loading the Dataset

In [10]:
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle
import tensorflow as tf 
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout, Multiply, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tqdm import tqdm
import time
from tensorflow.keras.mixed_precision import set_global_policy
import gc
from tensorflow.keras.models import load_model

In [ ]:
recipe2M = pd.read_csv('recipes_data.csv')

In [ ]:
recipe2M.head()

## Removing unnecessary columns & nulls

In [ ]:
recipe2M_cleaned=recipe2M.drop(columns=['link', 'source', 'site'], inplace=False)
recipe2M_cleaned.dropna()

## Removing recipes with directions contatining the word "step"

In [ ]:

recipe2M_cleaned = recipe2M_cleaned[~recipe2M_cleaned['directions'].str.contains('step', case=False, na=False)]
recipe2M_cleaned['title'].count()


## Removing recipes with at most 1 ingredient

In [ ]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['ingredients'].apply(lambda x: len([i for i in x if i.strip()]) <= 1)].index, inplace=True)
recipe2M_cleaned['title'].count()


## Removing recipes with instructions less than 10 characters

In [ ]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['directions'].apply(lambda x: not all(len(i.strip()) < 10 for i in x if i.strip()))].index, inplace=True)
recipe2M_cleaned['title'].count()

## Removing recipes with title less than 4 characters 

In [ ]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['title'].apply(lambda x: len(str(x)) < 4 if pd.notnull(x) else False)].index, inplace=True)
recipe2M_cleaned['title'].count()

## Extract raw ingredients

In [ ]:
recipes = recipe2M_cleaned

In [ ]:
# Tokenize a string into words
recipes['tokens'] = recipes['NER'].apply(word_tokenize)


In [ ]:
#adding customized stop words
irrelevant_words = {
    'fresh', 'frozen', 'thawed', 'raw', 'grated', 'diced', 'chopped', 'minced',
    'powdered', 'sliced', 'ground', 'cooked', 'boiled', 'roasted', 'steamed',
    'baked', 'fried', 'toasted', 'crushed', 'peeled', 'skinned', 'shredded',
    'melted', 'whipped', 'pinch', 'dash', 'handful', 'cup', 'tablespoon',
    'teaspoon', 'liter', 'ml', 'oz', 'lb', 'gram', 'kg', 'quart', 'optional',
    'to taste', 'as needed', 'prepared', 'ready-made', 'store-bought', 'homemade',
    'pre-cooked', 'large', 'small', 'medium', 'whole', 'half', 'quartered',
    'extra', 'light', 'dark', 'white', 'black', 'red', 'green', 'yellow',
    'brown', 'golden', 'sweet', 'bitter', 'spicy', 'mild', 'hot', 'cold',
    'water', 'broth', 'stock', 'sauce', 'seasoning', 'marinade','bite','size'
}
stop_words = set(stopwords.words('english'))
stop_words.update(irrelevant_words)

In [ ]:
lemmatizer = WordNetLemmatizer()

# Function to lemmatize nouns
def lemmatize(word, pos):
    if pos.startswith('NN'):  
        return lemmatizer.lemmatize(word, pos='n')
    else:
        return word  


In [ ]:
#apply stop words removal and lemmatization
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: [lemmatize(word.lower(), tag) for word, tag in nltk.pos_tag(x) if word.isalnum() and word.lower() not in stop_words]
)

In [ ]:
#filter uninque ingredients
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: list(set(x))
)

In [ ]:
print(recipes['tokens'])

In [ ]:
#adding ids to recipes
recipes['recipe_id'] = recipes.index + 1

In [ ]:
#reorder the columns
columns = ['recipe_id'] + [col for col in recipes.columns if col != 'recipe_id']
recipes = recipes[columns]

In [ ]:
recipes.rename(columns={'tokens': 'raw_ingredients'}, inplace=True)

In [ ]:
recipes.head()

# Extract Cooking Methods

In [ ]:
recipes.head()

In [ ]:
recipes['directions']

In [ ]:
cooking_methods_glossary = [
    "bake", "steam", "fry", "grill", "roast", "boil", "saut�", "poach", "broil", "braise",
    "stew", "smoke", "microwave", "blanch", "deep-fry", "barbecue", "sear", "pressure-cook",
    "simmer", "stir-fry","baste","batter","beat","blend","carmelize","chop","cream","cube",
    "cure","dice","dissolve","drain","fold","granish","grate","grease","julienne","knead",
    "marinate","mash","mince","parboil","pare","peel","pinch","pit","plump","preheat","puree",
    "reduce","saute","scald","sear","shred","sift","skim","slice","thaw","toss","whip"
]

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
cooking_methods = []

for directions in recipes['directions']:
    if pd.isna(directions):
        cooking_methods.append(None)
    else:
        # Tokenize words
        words = word_tokenize(directions.lower())
        # Remove stopwords and non-alphabetic tokens
        filtered_words = [word for word in words if word not in stop_words and word.isalpha()]

        methods = set(filtered_words).intersection(cooking_methods_glossary)

        cooking_methods.append(list(methods))

recipes['cooking_methods'] = cooking_methods

In [ ]:
recipes['cooking_methods'].head(20)

## Clean rating dataset

In [ ]:
train_rating= pd.read_csv('core-data-train_rating.csv')
test_rating = pd.read_csv('core-data-test_rating.csv')

In [ ]:
#mapping train data with correct recipe id
unique_old_ids = sorted(train_rating['recipe_id'].unique())
new_ids = recipes['recipe_id'].tolist() 

mapping = {old: new for old, new in zip(unique_old_ids, new_ids)}
train_rating['recipe_id'] = train_rating['recipe_id'].map(mapping)

In [ ]:
#mapping test data with correct recipe id
unique_old_ids = sorted(test_rating['recipe_id'].unique())
new_ids = recipes['recipe_id'].tolist()  

mapping = {old: new for old, new in zip(unique_old_ids, new_ids)}
test_rating['recipe_id'] = test_rating['recipe_id'].map(mapping)

In [ ]:
train_rating.drop(columns=['dateLastModified'], inplace=True)
test_rating.drop(columns=['dateLastModified'], inplace=True)

In [ ]:
recipes.to_csv('cleanedrecipes.csv', index=False)

In [ ]:
train_rating.to_csv('cleanedTrainRating.csv',index=False)
test_rating.to_csv('cleanedTestRating.csv',index=False)

# Models

In [2]:
cleaned_recipes = pd.read_csv('/kaggle/input/late-plate/cleanedrecipes.csv')
cleaned_train_rating = pd.read_csv('/kaggle/input/rating/cleanedTrainRating.csv')
cleaned_test_rating = pd.read_csv('/kaggle/input/rating/cleanedTestRating.csv')

In [ ]:
# Ensure every user-recipe pair is unique
cleaned_train_rating.duplicated(subset=['user_id' , 'recipe_id']).sum()

## Collaborative Filtering

In [29]:
train_df = cleaned_train_rating.copy()
test_df = cleaned_test_rating.copy()

# Filter users with at least 5 interactions
user_counts = train_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 5].index
train_df = train_df[train_df['user_id'].isin(valid_users)]
test_df = test_df[test_df['user_id'].isin(valid_users)]

# Create binary labels based on threshold
THRESHOLD = 2  # Ratings >= 3 are positive
train_df['label'] = (train_df['rating'] >= THRESHOLD).astype(int)
test_df['label'] = (test_df['rating'] >= THRESHOLD).astype(int)

# Split train into train and validation
train_data, val_data = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42,
    stratify=train_df['user_id']
)

# Combine all data for consistent indexing
combined = pd.concat([train_data, val_data, test_df])
combined['user_idx'] = combined['user_id'].astype('category').cat.codes
combined['recipe_idx'] = combined['recipe_id'].astype('category').cat.codes

# Split back to datasets
train_data = combined[combined.index.isin(train_data.index)]
val_data = combined[combined.index.isin(val_data.index)]
test_df = combined[combined.index.isin(test_df.index)]

# Get unique counts
n_users = combined['user_idx'].nunique()
n_recipes = combined['recipe_idx'].nunique()

print(f"Users: {n_users}, Recipes: {n_recipes}")
print(f"Train size: {len(train_data)}, Val size: {len(val_data)}, Test size: {len(test_df)}")


Users: 26263, Recipes: 36554
Train size: 601076, Val size: 150303, Test size: 324463


In [4]:
# ========== MODEL DEFINITION ==========
def build_neumf_model(n_users, n_recipes, embedding_dim=64):
    # Inputs
    user_input = Input(shape=(1,))
    recipe_input = Input(shape=(1,))
    
    # MF Path
    mf_user_embed = Embedding(n_users, embedding_dim, embeddings_regularizer=l2(0.001))(user_input)
    mf_recipe_embed = Embedding(n_recipes, embedding_dim, embeddings_regularizer=l2(0.001))(recipe_input)
    mf_user = Flatten()(mf_user_embed)
    mf_recipe = Flatten()(mf_recipe_embed)
    mf_vector = Multiply()([mf_user, mf_recipe])
    
    # MLP Path
    mlp_user_embed = Embedding(n_users, embedding_dim*2, embeddings_regularizer=l2(0.001))(user_input)
    mlp_recipe_embed = Embedding(n_recipes, embedding_dim*2, embeddings_regularizer=l2(0.001))(recipe_input)
    mlp_vector = Concatenate()([Flatten()(mlp_user_embed), Flatten()(mlp_recipe_embed)])
    mlp_vector = Dense(64, activation='relu')(mlp_vector)
    mlp_vector = BatchNormalization()(mlp_vector)
    mlp_vector = Dropout(0.3)(mlp_vector)
    
    # Combine paths
    concat = Concatenate()([mf_vector, mlp_vector])
    concat = Dense(32, activation='relu')(concat)
    
    # Output with sigmoid activation for binary classification
    output = Dense(1, activation='sigmoid')(concat)
    
    model = Model(inputs=[user_input, recipe_input], outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

# Build and train model
embedding_dim = 128
model = build_neumf_model(n_users, n_recipes, embedding_dim)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 1)              │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_1             │ (None, 1)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_2 (Embedding)   │ (None, 1, 256)         │      6,723,328 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_3 (Embedding)   │ (None, 1, 256)         │      9,357,824 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_2 (Flatten)       │ (None, 256)            │              0 │ embedding_2[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_3 (Flatten)       │ (None, 256)            │              0 │ embedding_3[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 512)            │              0 │ flatten_2[0][0],       │
│                           │                        │                │ flatten_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding (Embedding)     │ (None, 1, 128)         │      3,361,664 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_1 (Embedding)   │ (None, 1, 128)         │      4,678,912 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 64)             │         32,832 │ concatenate[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten (Flatten)         │ (None, 128)            │              0 │ embedding[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_1 (Flatten)       │ (None, 128)            │              0 │ embedding_1[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 64)             │            256 │ dense[0][0]            │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multiply (Multiply)       │ (None, 128)            │              0 │ flatten[0][0],         │
│                           │                        │                │ flatten_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 64)             │              0 │ batch_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_1             │ (None, 192)            │              0 │ multiply[0][0],        │
│ (Concatenate)             │                        │                │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 32)             │          6,176 │ concatenate_1[0][0]    │
├──────────────────────

 Total params: 24,161,025 (92.17 MB)

 Trainable params: 24,160,897 (92.17 MB)

 Non-trainable params: 128 (512.00 B)

In [5]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_auc', mode='max'),
    ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-5, verbose=1)
]

history = model.fit(
    [train_data['user_idx'], train_data['recipe_idx']],
    train_data['label'],
    batch_size=512,
    epochs=50,
    validation_data=([val_data['user_idx'], val_data['recipe_idx']], val_data['label']),
    callbacks=callbacks,
    verbose=1
)


Epoch 1/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - accuracy: 0.9661 - auc: 0.5616 - loss: 1.5410 - val_accuracy: 0.9855 - val_auc: 0.6487 - val_loss: 0.1583 - learning_rate: 0.0010
Epoch 2/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9859 - auc: 0.6850 - loss: 0.1473 - val_accuracy: 0.9855 - val_auc: 0.6883 - val_loss: 0.1305 - learning_rate: 0.0010
Epoch 3/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.9859 - auc: 0.7891 - loss: 0.1415 - val_accuracy: 0.9855 - val_auc: 0.7200 - val_loss: 0.1243 - learning_rate: 0.0010
Epoch 4/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9856 - auc: 0.8478 - loss: 0.1353 - val_accuracy: 0.9855 - val_auc: 0.7188 - val_loss: 0.1242 - learning_rate: 0.0010
Epoch 5/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.9855 - auc: 0.8745 - loss: 0.1312 - val_accuracy: 0.9855 - val_auc: 0.7128 - val_loss: 0.1315 - learning_rate: 0.0010
Epoch 6/50
1172/1174 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.

In [71]:
user_seen = combined.groupby('user_idx')['recipe_idx'].apply(set).to_dict()
all_items = np.array(combined['recipe_idx'].unique())

# Only consider positive interactions for test
test_positives = test_df[test_df['label'] == 1]
test_users = test_positives['user_idx'].unique()
def evaluate_user_level(model, k_list=[5, 10, 20]):
    
    test_user_positives = test_positives.groupby('user_idx')['recipe_idx'].apply(set).to_dict()
    combined_all = pd.concat([train_data, val_data])
    user_seen = combined_all.groupby('user_idx')['recipe_idx'].apply(set).to_dict()
    
    precision_at_k = {k: [] for k in k_list}
    recall_at_k = {k: [] for k in k_list}
    ndcg_at_k = {k: [] for k in k_list}
    
    for user in tqdm(test_users, desc="User-Level Evaluation"):
        # Get user's positive items in test set
        true_positives = test_user_positives.get(user, set())
        if not true_positives:
            continue
            
        # Get candidate items (unseen by user)
        seen = user_seen.get(user, set())
        candidate_items = list(set(all_items) - seen)
        
        negatives = list(set(all_items) - seen - true_positives)  
        if len(negatives) > 50: 
            negatives = np.random.choice(negatives, 50, replace=False)
        test_items = list(true_positives) + list(negatives)
            
        
        
        users_arr = np.full(len(test_items), user, dtype=np.int32)
        items_arr = np.array(test_items, dtype=np.int32)
        
        # Predict scores
        scores = model.predict([users_arr, items_arr], 
                              batch_size=1024, 
                              verbose=0).flatten()
        
        # Create item-score mapping
        item_scores = dict(zip(test_items, scores))
        
        # Rank all items by score (highest first)
        ranked_items = [item for item, score in 
                       sorted(item_scores.items(), key=lambda x: x[1], reverse=True)]
       
        # Calculate metrics for each K
        for k in k_list:
            
            top_k = ranked_items[:k]
            
            # Calculate true positives in top-k
            hits = len(set(top_k) & true_positives)
            precision = hits / k
            precision_at_k[k].append(precision)
            
            
            recall = hits / len(true_positives) if len(true_positives) > 0 else 0
            recall_at_k[k].append(recall)
            
            # NDCG@k
            dcg = 0
            for i, item in enumerate(top_k, 1):
                if item in true_positives:
                    dcg += 1 / np.log2(i + 1)
                    
            # Ideal DCG
            ideal_top_k = min(len(true_positives), k)
            idcg = sum(1 / np.log2(i + 1) for i in range(1, ideal_top_k + 1))
            
            ndcg = dcg / idcg if idcg > 0 else 0
            ndcg_at_k[k].append(ndcg)
    
    
    results = {}
    for k in k_list:
        results[f'Precision@{k}'] = np.mean(precision_at_k[k])
        results[f'Recall@{k}'] = np.mean(recall_at_k[k])
        results[f'NDCG@{k}'] = np.mean(ndcg_at_k[k])   
    return results

# Run user-level evaluation
top_k = [5, 10, 20]
user_level_results = evaluate_user_level(model, k_list=top_k)

print("\n===== User-Level Evaluation Results =====")
print(f"Evaluated {len(test_users)} users")
print("-" * 40)
for k in top_k:
    print(f"** Top-{k} **")
    print(f"Precision@{k}: {user_level_results[f'Precision@{k}']:.4f}")
    print(f"Recall@{k}: {user_level_results[f'Recall@{k}']:.4f}")
    print(f"NDCG@{k}: {user_level_results[f'NDCG@{k}']:.4f}")
    print("-" * 40)


User-Level Evaluation: 100%|██████████| 26119/26119 [38:48<00:00, 11.22it/s]  



===== User-Level Evaluation Results =====
Evaluated 26119 users
----------------------------------------
** Top-5 **
Precision@5: 0.3177
Recall@5: 0.2309
NDCG@5: 0.3712
----------------------------------------
** Top-10 **
Precision@10: 0.2836
Recall@10: 0.3906
NDCG@10: 0.4006
----------------------------------------
** Top-20 **
Precision@20: 0.2383
Recall@20: 0.6104
NDCG@20: 0.4590
----------------------------------------


In [38]:
#rmse
test_pred_probs = model.predict(
    [test_df['user_idx'], test_df['recipe_idx']],
    batch_size=1024,
    verbose=1
).flatten()


binary_rmse = np.sqrt(mean_squared_error(test_df['label'], test_pred_probs))
print(f"Binary RMSE: {binary_rmse:.4f}")

317/317 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Binary RMSE: 0.1135


In [6]:
model.save("neumf_model.h5")

In [7]:
# Build mapping dictionaries
user2idx = dict(zip(combined['user_id'], combined['user_idx']))
idx2recipe = dict(zip(combined['recipe_idx'], combined['recipe_id']))
user_seen = combined.groupby('user_idx')['recipe_idx'].apply(set).to_dict()

all_items = np.array(combined['recipe_idx'].unique())

with open('mappings.pkl', 'wb') as f:
    pickle.dump({'user2idx': user2idx, 'idx2recipe': idx2recipe, 'user_seen': user_seen}, f)



In [8]:
MODEL_PATH = "/kaggle/working/neumf_model.h5"
MAPPINGS_PATH = "/kaggle/working/mappings.pkl"

def recommend_recipes_CF(user_id, top_k=10):
    
    model = load_model(MODEL_PATH)

    
    with open(MAPPINGS_PATH, 'rb') as f:
        data = pickle.load(f)
        user2idx = data['user2idx']
        idx2recipe = data['idx2recipe']
        user_seen = data['user_seen']

    all_items = np.array(list(set().union(*user_seen.values())))  

    # Check user
    if user_id not in user2idx:
        return []  

    user_idx = user2idx[user_id]
    seen_items = user_seen.get(user_idx, set())
    candidate_items = np.setdiff1d(all_items, list(seen_items))

    if len(candidate_items) == 0:
        return []  

    # Predict
    user_input = np.full(len(candidate_items), user_idx, dtype=np.int32)
    predictions = model.predict([user_input, candidate_items], batch_size=512, verbose=0).flatten()

    # Get top-k indices and scores
    top_k_indices = np.argsort(predictions)[-top_k:][::-1]
    top_recipe_idxs = candidate_items[top_k_indices]
    top_scores = predictions[top_k_indices]
    
    
    recommendations = {
        idx2recipe[i]: float(score)  
        for i, score in zip(top_recipe_idxs, top_scores)
    }
    
    return recommendations

In [11]:
recommended = recommend_recipes_CF(user_id=7334589, top_k=10)
print("Recommended Recipes:", recommended)


Recommended Recipes: {15967: 0.9991760849952698, 25797: 0.999079704284668, 5769: 0.9990731477737427, 6134: 0.9990481734275818, 13846: 0.9990310668945312, 5282: 0.9990202188491821, 395: 0.9990167617797852, 14460: 0.9989838004112244, 8045: 0.9989365935325623, 33: 0.9989325404167175}


## Popularity Based

In [12]:
# Calculate average rating and count
popularity_avg = cleaned_train_rating.groupby('recipe_id')['rating'].agg(['mean', 'count']).reset_index()
popularity_avg.columns = ['recipe_id', 'avg_rating', 'num_ratings']

# Filter recipes with at least 3 ratings 
min_ratings = 3
popularity_avg_filtered = popularity_avg[popularity_avg['num_ratings'] >= min_ratings]

#Rank by highest average rating
top_rated = popularity_avg_filtered.sort_values(by='avg_rating', ascending=False)
print("\nHighest Average-Rated Recipes (with min 2 ratings):")
print(top_rated.head())


Highest Average-Rated Recipes (with min 2 ratings):
       recipe_id  avg_rating  num_ratings
28873      28899         5.0            4
28936      28962         5.0            3
28871      28897         5.0            3
29088      29114         5.0            4
28826      28852         5.0            3


In [13]:
#Weighted score 
min_ratings_for_weight = 1 
popularity_avg['weighted_score'] = (popularity_avg['avg_rating'] * popularity_avg['num_ratings']) / (popularity_avg['num_ratings'] + min_ratings_for_weight)

#Rank by weighted score
top_hybrid = popularity_avg.sort_values(by='weighted_score', ascending=False)
print("\nHybrid Popularity (Weighted Score):")
print(top_hybrid.head())


Hybrid Popularity (Weighted Score):
       recipe_id  avg_rating  num_ratings  weighted_score
27167      27192    4.952941           85        4.895349
19768      19790    4.906897          290        4.890034
5584        5592    4.901163          344        4.886957
19090      19111    4.904762          252        4.885375
4367        4373    4.913669          139        4.878571


In [14]:
def recommend_popular_recipes(popularity_df, n=10):
    return popularity_df['recipe_id'].tolist()[:n]

recommendations = recommend_popular_recipes(top_hybrid)
print("\nTop Recommendations:", recommendations)


Top Recommendations: [27192, 19790, 5592, 19111, 4373, 21627, 2280, 9762, 6162, 23855]


## Content Based

In [15]:
np.random.seed(42)  
tf.random.set_seed(42)  

physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available:", len(physical_devices))
if len(physical_devices) > 0:
    print("GPU detected:", physical_devices)
else:
    print("No GPU detected. Running on CPU.")

set_global_policy('mixed_float16')  

cleaned_recipes['raw_ingredients'] = cleaned_recipes['raw_ingredients'].fillna('')
cleaned_recipes['cooking_methods'] = cleaned_recipes['cooking_methods'].fillna('')

def parse_list(x):
    if isinstance(x, str) and x.startswith('['):
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, list) else []
        except:
            return []
    return []

cleaned_recipes['raw_ingredients'] = cleaned_recipes['raw_ingredients'].apply(parse_list)
cleaned_recipes['cooking_methods'] = cleaned_recipes['cooking_methods'].apply(parse_list)

cleaned_recipes = cleaned_recipes[
    ~((cleaned_recipes['raw_ingredients'].apply(len) == 0) & (cleaned_recipes['cooking_methods'].apply(len) == 0)) &
    (cleaned_recipes['cooking_methods'].apply(len) > 0)
]
cleaned_recipes = cleaned_recipes.reset_index(drop=True)

cleaned_recipes['combined_text'] = cleaned_recipes['raw_ingredients'].apply(lambda x: ' '.join(x)) + ' ' + cleaned_recipes['cooking_methods'].apply(lambda x: ' '.join(x))

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
recipe_features = tfidf.fit_transform(cleaned_recipes['combined_text'])
recipe_ids = cleaned_recipes['recipe_id'].values

joblib.dump(tfidf, '/kaggle/working/tfidf_vectorizer.pkl')
joblib.dump(recipe_features, '/kaggle/working/recipe_features_sparse.pkl')



Num GPUs Available: 2
GPU detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


['/kaggle/working/recipe_features_sparse.pkl']

In [16]:
recipe_id_to_index = {rid: idx for idx, rid in enumerate(recipe_ids)}

def batch_generator(sparse_matrix, ratings, recipe_id_to_index, batch_size):
    n_samples = len(ratings)
    indices = np.arange(n_samples)
    np.random.seed(42) 
    np.random.shuffle(indices)
    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        batch_indices = indices[start:end]
        batch_X = []
        batch_y = []
        for idx in batch_indices:
            row = ratings.iloc[idx]
            if row['recipe_id'] in recipe_id_to_index:
                dense_vector = sparse_matrix[recipe_id_to_index[row['recipe_id']]].toarray().flatten()
                batch_X.append(dense_vector)
                batch_y.append(row['rating'])
        if batch_X:  
            yield np.array(batch_X, dtype=np.float32), np.array(batch_y, dtype=np.float32)


In [17]:
model2 = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(5000,)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='linear', dtype='float32')  # Match mixed precision
])
model2.compile(optimizer='adam', loss='mse', metrics=['mae'])

batch_size = 128  
steps_per_epoch = max(1, len(cleaned_train_rating) // batch_size)
model2.fit(batch_generator(recipe_features, cleaned_train_rating, recipe_id_to_index, batch_size),
          steps_per_epoch=steps_per_epoch, epochs=50, verbose=1)

gc.collect()

Epoch 1/50


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5288/5288 ━━━━━━━━━━━━━━━━━━━━ 177s 33ms/step - loss: 1.5897 - mae: 0.9060
Epoch 2/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 3s 5us/step - loss: 0.7561 - mae: 0.6432    
Epoch 3/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 4/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 5/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 6/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 7/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 8/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 9/50


/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 10/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 11/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 12/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 6us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 13/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 14/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 15/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 16/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 17/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 6us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 18/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00
Epoch 19/50
5288/5288 ━━━━━━━━━━━━━━━━━━━━ 0s 5us/step - loss: 0.0000e+00 - mae: 0.0000e+00


8411

In [81]:
test_generator = batch_generator(recipe_features, cleaned_test_rating, recipe_id_to_index, batch_size)
predictions = []
y_test = []
for X_batch, y_batch in test_generator:
    batch_pred = model.predict(X_batch, verbose=0)
    predictions.extend(batch_pred.flatten())
    y_test.extend(y_batch)
predictions = np.array(predictions)
y_test = np.array(y_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"Test RMSE: {rmse}")

Test RMSE: 0.8604246973991394


In [18]:
def precompute_user_profiles(ratings, recipe_features, recipe_id_to_index):
    user_profiles = {}
    for user_id in ratings['user_id'].unique():
        user_ratings = ratings[ratings['user_id'] == user_id]
        high_rated = user_ratings[user_ratings['rating'] >= 4]
        if not high_rated.empty:
            rated_recipe_ids = high_rated['recipe_id'].values
            rated_indices = [recipe_id_to_index[rid] for rid in rated_recipe_ids if rid in recipe_id_to_index]
            if rated_indices:
                user_profile = recipe_features[rated_indices].mean(axis=0).A1
                user_profiles[user_id] = user_profile
    return user_profiles

def evaluate_precision_recall(ratings, recipe_features, recipe_ids, cleaned_recipes, recipe_id_to_index, top_k=5, sample_size=1000):
    precision_scores = []
    recall_scores = []
    
    users = ratings['user_id'].unique()
    if sample_size and sample_size < len(users):
        users = np.random.choice(users, size=sample_size, replace=False)
    
    user_profiles = precompute_user_profiles(ratings, recipe_features, recipe_id_to_index)
    
    for user_id in tqdm(users, desc="Evaluating users"):
        user_ratings = ratings[ratings['user_id'] == user_id]
        relevant_items = set(user_ratings[user_ratings['rating'] >= 4]['recipe_id'].values)
        
        if not relevant_items:
            continue
        
        if user_id in user_profiles:
            user_profile = user_profiles[user_id]
            similarities = cosine_similarity(recipe_features, user_profile.reshape(1, -1)).flatten()
            top_indices = np.argsort(similarities)[::-1][:top_k]
            recommended_ids = set(recipe_ids[top_indices])
            
            true_positives = len(recommended_ids & relevant_items)
            precision = true_positives / top_k if top_k > 0 else 0
            recall = true_positives / len(relevant_items) if len(relevant_items) > 0 else 0
            
            precision_scores.append(precision)
            recall_scores.append(recall)
    
    avg_precision = np.mean(precision_scores) if precision_scores else 0
    avg_recall = np.mean(recall_scores) if recall_scores else 0
    return avg_precision, avg_recall

In [19]:
def create_user_profile(user_id, ratings, recipe_features, recipe_ids):
    user_ratings = ratings[ratings['user_id'] == user_id]
    high_rated = user_ratings[user_ratings['rating'] >= 4]
    if not high_rated.empty:
        rated_recipe_ids = high_rated['recipe_id'].values
        rated_indices = [recipe_id_to_index[rid] for rid in rated_recipe_ids if rid in recipe_id_to_index]
        if rated_indices:
            user_profile = recipe_features[rated_indices].mean(axis=0).A1
            return user_profile
    return None

def recommend_recipes(user_id, ratings, recipe_features, recipe_ids, cleaned_recipes, top_k=5):
    user_profile = create_user_profile(user_id, ratings, recipe_features, recipe_ids)
    if user_profile is not None:
        similarities = cosine_similarity(recipe_features, user_profile.reshape(1, -1)).flatten()
        top_indices = np.argsort(similarities)[::-1][:top_k]
        recommended_recipe_ids = recipe_ids[top_indices]
        return cleaned_recipes[cleaned_recipes['recipe_id'].isin(recommended_recipe_ids)][['recipe_id', 'raw_ingredients', 'cooking_methods']]
    return "User not found or no high-rated recipes."


gc.collect()

user_id = 5215572 
recommendations = recommend_recipes(user_id, cleaned_train_rating, recipe_features, recipe_ids, cleaned_recipes)
print("Recommendations for user", user_id)
print(recommendations)

model.save('/kaggle/working/recipe_rating_model.h5')

Recommendations for user 5215572
         recipe_id                                    raw_ingredients  \
7570          9088  [sorghum, salt, cinnamon, sugar, egg, milk, pu...   
21131        25264  [flour, wine, pepper, salt, mushroom, pearl, s...   
73253        87568  [sorghum, cinnamon, sugar, egg, milk, nutmeg, ...   
480659      575189  [sorghum, flour, cinnamon, sugar, oil, egg, milk]   
1447650    1684867  [sorghum, flour, salt, cinnamon, soda, shorten...   

             cooking_methods  
7570                  [bake]  
21131    [saute, skim, bake]  
73253                 [bake]  
480659                [bake]  
1447650        [cream, bake]  


In [84]:
precision, recall = evaluate_precision_recall(
    cleaned_test_rating, recipe_features, recipe_ids, cleaned_recipes, 
    recipe_id_to_index, top_k=5, sample_size=5000
)
print(f"Precision@5: {precision:.4f}")
print(f"Recall@5: {recall:.4f}")

precision_10, recall_10 = evaluate_precision_recall(
    cleaned_test_rating, recipe_features, recipe_ids, cleaned_recipes, 
    recipe_id_to_index, top_k=10, sample_size=5000
)
print(f"Precision@10: {precision_10:.4f}")
print(f"Recall@10: {recall_10:.4f}")

precision_20, recall_20 = evaluate_precision_recall(
    cleaned_test_rating, recipe_features, recipe_ids, cleaned_recipes, 
    recipe_id_to_index, top_k=20, sample_size=5000
)
print(f"Precision@20: {precision_20:.4f}")
print(f"Recall@20: {recall_20:.4f}") 

Evaluating users:  33%|███▎      | 1630/5000 [08:35<17:45,  3.16it/s]


KeyboardInterrupt: 

In [20]:
def parse_list(x):
    import ast
    if isinstance(x, str) and x.startswith('['):
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, list) else []
        except:
            return []
    return []

def get_recommendations_CB(user_id, top_n=10):
    vectorizer_path = '/kaggle/working/tfidf_vectorizer.pkl'
    features_path = '/kaggle/working/recipe_features_sparse.pkl'
    tfidf = joblib.load(vectorizer_path)
    recipe_features = joblib.load(features_path) 

    cleaned_recipes['raw_ingredients'] = cleaned_recipes['raw_ingredients'].fillna('').apply(parse_list)
    cleaned_recipes['cooking_methods'] = cleaned_recipes['cooking_methods'].fillna('').apply(parse_list)
    cleaned_recipes['combined_text'] = cleaned_recipes['raw_ingredients'].apply(lambda x: ' '.join(x)) + ' ' + cleaned_recipes['cooking_methods'].apply(lambda x: ' '.join(x))
    recipe_ids = cleaned_recipes['recipe_id'].values
    recipe_id_to_index = {rid: idx for idx, rid in enumerate(recipe_ids)}

    user_ratings = cleaned_train_rating[cleaned_train_rating['user_id'] == user_id]
    high_rated = user_ratings[user_ratings['rating'] >= 4]
    if high_rated.empty:
        return {}

    rated_recipe_ids = high_rated['recipe_id'].values
    rated_indices = [recipe_id_to_index[rid] for rid in rated_recipe_ids if rid in recipe_id_to_index]
    if not rated_indices:
        return {}

    user_profile = recipe_features[rated_indices].mean(axis=0).A1  # dense vector

    similarity_scores = cosine_similarity(user_profile.reshape(1, -1), recipe_features).flatten()
    top_indices = np.argpartition(similarity_scores, -top_n)[-top_n:]
    top_indices = top_indices[np.argsort(similarity_scores[top_indices])[::-1]] 

    top_scores = similarity_scores[top_indices]
    top_recipe_ids = recipe_ids[top_indices]

    return dict(zip(top_recipe_ids, top_scores))


In [21]:
user_id = 5215572

recommendations = get_recommendations_CB(user_id)

print("Top recommendations:", recommendations)


Top recommendations: {25264: 0.7191659402269119, 9088: 0.7191659402269118, 87568: 0.6580643891710369, 575189: 0.6211817190790604, 1684867: 0.5935123137478785, 185838: 0.5784402012934902, 714738: 0.5756587824166856, 307335: 0.5749619427841398, 399525: 0.5726071653373676, 106569: 0.5601666889753284}


## Hybrid Filtering